In [1]:
import sys
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

Project directory set to: D:\dev\dermascan


In [2]:
from pipeline.pipeline_orchestrator import PipelineOrchestrator
from pipeline.dataclasses.pipeline_config import PipelineConfig, BranchConfig, ModelConfig, TrainerConfig, TrainValTestSplitRatio, AugmentationConfig, CustomSplitConfig
from pipeline.dataclasses.enums import ModelType, FilterType, DatasetLoaderType, AugmentationType, LossType
from pipeline.dataclasses.training_configs import AdamWConfig, CosineAnnealingLRConfig

In [3]:
pipeline_config = PipelineConfig(
    pipeline_id="threshold_test",
    data_path=proj_dir / "data",
    split_config=CustomSplitConfig(
        train_csv_path=Path(proj_dir / "data" / "custom" / "train_df.csv"),
        val_csv_path=Path(proj_dir / "data" / "custom" / "val_df.csv"),
        test_csv_path=Path(proj_dir / "data" / "custom" / "test_df.csv"),
    ),
    seed=42,
    branches=[
        BranchConfig(
            branch_id="EFFICIENTNET_V2_S",
            sampler_weights=None,
            branch_model_config=ModelConfig(
                model_type=ModelType.EFFICIENTNET_V2_S,
                filter_type=FilterType.RGB,
                num_classes=1,
                pretrained=True,
                freeze_weights=False
            ),
            trainer_config=TrainerConfig(
                batch_size=12,
                num_epochs=2,
                learning_rate=3e-4,
                loss_type=LossType.BCE_WITH_LOGITS,
                pos_weight=1.0,
                class_weights=None,
                optimizer=AdamWConfig(weight_decay=1e-4),
                scheduler=CosineAnnealingLRConfig(t_max=30, eta_min=1e-5),
            ),
            dataset_loader=DatasetLoaderType.RGB,
            augmentation=AugmentationConfig(
                train=[
                    AugmentationType.RANDOM_RESIZE_CROP,
                    AugmentationType.RANDOM_HORIZONTAL_FLIP,
                    AugmentationType.RANDOM_VERTICAL_FLIP,
                    AugmentationType.RANDOM_ROTATION,
                    AugmentationType.COLOR_JITTER,
                ],
                val=[
                    AugmentationType.RESIZE
                ]
            ),
        ),
    ]
)

In [4]:
pipeline_runner = PipelineOrchestrator(
    pipeline_config=pipeline_config,
    project_root_path=proj_dir,
)

In [5]:
run = pipeline_runner.run()

Starting pipeline run: threshold_test
Pipeline run path: D:\dev\dermascan\results\threshold_test_2026-06-05_19-57
----------
[BranchRunner] Starting run for branch: EFFICIENTNET_V2_S


Epochs:   0%|          | 0/2 [00:00<?, ?it/s]

Training:   0%|          | 0/489 [00:17<?, ?it/s]

Training:   0%|          | 0/489 [00:00<?, ?it/s]

Best F1 macro score found: 0.7745 at epoch 1
Optimal threshold found: 0.1


2026/06/05 20:29:00 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/05 20:29:01 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/05 20:29:10 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the loca